In [58]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# 데이터 로드
train_df = pd.read_csv(
    '../data/creditcard.csv'
)

In [59]:
train_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 284807 entries, 0 to 284806
Data columns (total 31 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Time    284807 non-null  float64
 1   V1      284807 non-null  float64
 2   V2      284807 non-null  float64
 3   V3      284807 non-null  float64
 4   V4      284807 non-null  float64
 5   V5      284807 non-null  float64
 6   V6      284807 non-null  float64
 7   V7      284807 non-null  float64
 8   V8      284807 non-null  float64
 9   V9      284807 non-null  float64
 10  V10     284807 non-null  float64
 11  V11     284807 non-null  float64
 12  V12     284807 non-null  float64
 13  V13     284807 non-null  float64
 14  V14     284807 non-null  float64
 15  V15     284807 non-null  float64
 16  V16     284807 non-null  float64
 17  V17     284807 non-null  float64
 18  V18     284807 non-null  float64
 19  V19     284807 non-null  float64
 20  V20     284807 non-null  float64
 21  V21     284807 non-nu

In [60]:
pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:.2f}'.format

In [61]:
train_df.head(10)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.00,-1.36,-0.07,2.54,1.38,-0.34,0.46,0.24,0.10,0.36,0.09,-0.55,-0.62,-0.99,-0.31,1.47,-0.47,0.21,0.03,0.40,0.25,-0.02,0.28,-0.11,0.07,0.13,-0.19,0.13,-0.02,149.62,0
1,0.00,1.19,0.27,0.17,0.45,0.06,-0.08,-0.08,0.09,-0.26,-0.17,1.61,1.07,0.49,-0.14,0.64,0.46,-0.11,-0.18,-0.15,-0.07,-0.23,-0.64,0.10,-0.34,0.17,0.13,-0.01,0.01,2.69,0
2,1.00,-1.36,-1.34,1.77,0.38,-0.50,1.80,0.79,0.25,-1.51,0.21,0.62,0.07,0.72,-0.17,2.35,-2.89,1.11,-0.12,-2.26,0.52,0.25,0.77,0.91,-0.69,-0.33,-0.14,-0.06,-0.06,378.66,0
3,1.00,-0.97,-0.19,1.79,-0.86,-0.01,1.25,0.24,0.38,-1.39,-0.05,-0.23,0.18,0.51,-0.29,-0.63,-1.06,-0.68,1.97,-1.23,-0.21,-0.11,0.01,-0.19,-1.18,0.65,-0.22,0.06,0.06,123.50,0
4,2.00,-1.16,0.88,1.55,0.40,-0.41,0.10,0.59,-0.27,0.82,0.75,-0.82,0.54,1.35,-1.12,0.18,-0.45,-0.24,-0.04,0.80,0.41,-0.01,0.80,-0.14,0.14,-0.21,0.50,0.22,0.22,69.99,0
5,2.00,-0.43,0.96,1.14,-0.17,0.42,-0.03,0.48,0.26,-0.57,-0.37,1.34,0.36,-0.36,-0.14,0.52,0.40,-0.06,0.07,-0.03,0.08,-0.21,-0.56,-0.03,-0.37,-0.23,0.11,0.25,0.08,3.67,0
6,4.00,1.23,0.14,0.05,1.20,0.19,0.27,-0.01,0.08,0.46,-0.10,-1.42,-0.15,-0.75,0.17,0.05,-0.44,0.00,-0.61,-0.05,-0.22,-0.17,-0.27,-0.15,-0.78,0.75,-0.26,0.03,0.01,4.99,0
7,7.00,-0.64,1.42,1.07,-0.49,0.95,0.43,1.12,-3.81,0.62,1.25,-0.62,0.29,1.76,-1.32,0.69,-0.08,-1.22,-0.36,0.32,-0.16,1.94,-1.02,0.06,-0.65,-0.42,-0.05,-1.21,-1.09,40.80,0
8,7.00,-0.89,0.29,-0.11,-0.27,2.67,3.72,0.37,0.85,-0.39,-0.41,-0.71,-0.11,-0.29,0.07,-0.33,-0.21,-0.50,0.12,0.57,0.05,-0.07,-0.27,-0.20,1.01,0.37,-0.38,0.01,0.14,93.20,0
9,9.00,-0.34,1.12,1.04,-0.22,0.50,-0.25,0.65,0.07,-0.74,-0.37,1.02,0.84,1.01,-0.44,0.15,0.74,-0.54,0.48,0.45,0.20,-0.25,-0.63,-0.12,-0.39,-0.07,0.09,0.25,0.08,3.68,0


In [62]:
train_df[train_df['Class']==1].head(10)

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
541,406.00,-2.31,1.95,-1.61,4.00,-0.52,-1.43,-2.54,1.39,-2.77,-2.77,3.20,-2.90,-0.60,-4.29,0.39,-1.14,-2.83,-0.02,0.42,0.13,0.52,-0.04,-0.47,0.32,0.04,0.18,0.26,-0.14,0.00,1
623,472.00,-3.04,-3.16,1.09,2.29,1.36,-1.06,0.33,-0.07,-0.27,-0.84,-0.41,-0.50,0.68,-1.69,2.00,0.67,0.60,1.73,0.28,2.10,0.66,0.44,1.38,-0.29,0.28,-0.15,-0.25,0.04,529.00,1
4920,4462.00,-2.30,1.76,-0.36,2.33,-0.82,-0.08,0.56,-0.40,-0.24,-1.53,2.03,-6.56,0.02,-1.47,-0.70,-2.28,-4.78,-2.62,-1.33,-0.43,-0.29,-0.93,0.17,-0.09,-0.16,-0.54,0.04,-0.15,239.93,1
6108,6986.00,-4.40,1.36,-2.59,2.68,-1.13,-1.71,-3.50,-0.25,-0.25,-4.80,4.90,-10.91,0.18,-6.77,-0.01,-7.36,-12.60,-5.13,0.31,-0.17,0.57,0.18,-0.44,-0.05,0.25,-0.66,-0.83,0.85,59.00,1
6329,7519.00,1.23,3.02,-4.30,4.73,3.62,-1.36,1.71,-0.50,-1.28,-2.45,2.10,-4.61,1.46,-6.08,-0.34,2.58,6.74,3.04,-2.72,0.01,-0.38,-0.70,-0.66,-1.63,1.49,0.57,-0.01,0.15,1.00,1
6331,7526.00,0.01,4.14,-6.24,6.68,0.77,-3.35,-1.63,0.15,-2.80,-6.19,5.66,-9.85,-0.31,-10.69,-0.64,-2.04,-1.13,0.12,-1.93,0.49,0.36,-0.61,-0.54,0.13,1.49,0.51,0.74,0.51,1.00,1
6334,7535.00,0.03,4.13,-6.56,6.35,1.33,-2.51,-1.69,0.30,-3.14,-6.05,6.75,-8.95,0.70,-10.73,-1.38,-1.64,-1.75,0.78,-1.33,0.59,0.37,-0.58,-0.67,-0.76,1.61,0.54,0.74,0.50,1.00,1
6336,7543.00,0.33,3.71,-5.78,6.08,1.67,-2.42,-0.81,0.13,-2.21,-5.13,4.56,-8.87,-0.80,-9.18,-0.26,-0.87,1.31,0.77,-2.37,0.27,0.16,-0.65,-0.55,-0.72,1.42,0.56,0.53,0.40,1.00,1
6338,7551.00,0.32,3.81,-5.62,6.05,1.55,-2.65,-0.75,0.06,-2.68,-4.96,6.44,-7.52,0.39,-9.25,-1.37,-0.50,0.78,1.49,-1.81,0.39,0.21,-0.51,-0.58,-0.22,1.47,0.49,0.52,0.40,1.00,1
6427,7610.00,0.73,2.30,-5.33,4.01,-1.73,-1.73,-3.97,1.06,-0.49,-4.62,5.59,-7.15,1.68,-6.21,0.50,-3.60,-4.83,-0.65,2.25,0.50,0.59,0.11,0.60,-0.36,-1.84,0.35,0.59,0.10,1.00,1


In [63]:
train_df.describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00,284807.00
mean,94813.86,0.00,0.00,-0.00,0.00,0.00,0.00,-0.00,0.00,-0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,0.00,0.00,0.00,0.00,-0.00,-0.00,88.35,0.00
std,47488.15,1.96,1.65,1.52,1.42,1.38,1.33,1.24,1.19,1.10,1.09,1.02,1.00,1.00,0.96,0.92,0.88,0.85,0.84,0.81,0.77,0.73,0.73,0.62,0.61,0.52,0.48,0.40,0.33,250.12,0.04
min,0.00,-56.41,-72.72,-48.33,-5.68,-113.74,-26.16,-43.56,-73.22,-13.43,-24.59,-4.80,-18.68,-5.79,-19.21,-4.50,-14.13,-25.16,-9.50,-7.21,-54.50,-34.83,-10.93,-44.81,-2.84,-10.30,-2.60,-22.57,-15.43,0.00,0.00
25%,54201.50,-0.92,-0.60,-0.89,-0.85,-0.69,-0.77,-0.55,-0.21,-0.64,-0.54,-0.76,-0.41,-0.65,-0.43,-0.58,-0.47,-0.48,-0.50,-0.46,-0.21,-0.23,-0.54,-0.16,-0.35,-0.32,-0.33,-0.07,-0.05,5.60,0.00
50%,84692.00,0.02,0.07,0.18,-0.02,-0.05,-0.27,0.04,0.02,-0.05,-0.09,-0.03,0.14,-0.01,0.05,0.05,0.07,-0.07,-0.00,0.00,-0.06,-0.03,0.01,-0.01,0.04,0.02,-0.05,0.00,0.01,22.00,0.00
75%,139320.50,1.32,0.80,1.03,0.74,0.61,0.40,0.57,0.33,0.60,0.45,0.74,0.62,0.66,0.49,0.65,0.52,0.40,0.50,0.46,0.13,0.19,0.53,0.15,0.44,0.35,0.24,0.09,0.08,77.16,0.00
max,172792.00,2.45,22.06,9.38,16.88,34.80,73.30,120.59,20.01,15.59,23.75,12.02,7.85,7.13,10.53,8.88,17.32,9.25,5.04,5.59,39.42,27.20,10.50,22.53,4.58,7.52,3.52,31.61,33.85,25691.16,1.00


In [64]:
train_df[train_df['Class']==1].describe()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00,492.00
mean,80746.81,-4.77,3.62,-7.03,4.54,-3.15,-1.40,-5.57,0.57,-2.58,-5.68,3.80,-6.26,-0.11,-6.97,-0.09,-4.14,-6.67,-2.25,0.68,0.37,0.71,0.01,-0.04,-0.11,0.04,0.05,0.17,0.08,122.21,1.00
std,47835.37,6.78,4.29,7.11,2.87,5.37,1.86,7.21,6.80,2.50,4.90,2.68,4.65,1.10,4.28,1.05,3.87,6.97,2.90,1.54,1.35,3.87,1.49,1.58,0.52,0.80,0.47,1.38,0.55,256.68,0.00
min,406.00,-30.55,-8.40,-31.10,-1.31,-22.11,-6.41,-43.56,-41.04,-13.43,-24.59,-1.70,-18.68,-3.13,-19.21,-4.50,-14.13,-25.16,-9.50,-3.68,-4.13,-22.80,-8.89,-19.25,-2.03,-4.78,-1.15,-7.26,-1.87,0.00,1.00
25%,41241.50,-6.04,1.19,-8.64,2.37,-4.79,-2.50,-7.97,-0.20,-3.87,-7.76,1.97,-8.69,-0.98,-9.69,-0.64,-6.56,-11.95,-4.66,-0.30,-0.17,0.04,-0.53,-0.34,-0.44,-0.31,-0.26,-0.02,-0.11,1.00,1.00
50%,75568.50,-2.34,2.72,-5.08,4.18,-1.52,-1.42,-3.03,0.62,-2.21,-4.58,3.59,-5.50,-0.07,-6.73,-0.06,-3.55,-5.30,-1.66,0.65,0.28,0.59,0.05,-0.07,-0.06,0.09,0.00,0.39,0.15,9.25,1.00
75%,128483.00,-0.42,4.97,-2.28,6.35,0.21,-0.41,-0.95,1.76,-0.79,-2.61,5.31,-2.97,0.67,-4.28,0.61,-1.23,-1.34,0.09,1.65,0.82,1.24,0.62,0.31,0.29,0.46,0.40,0.83,0.38,105.89,1.00
max,170348.00,2.13,22.06,2.25,12.11,11.10,6.47,5.80,20.01,3.35,4.03,12.02,1.38,2.82,3.44,2.47,3.14,6.74,3.79,5.23,11.06,27.20,8.36,5.47,1.09,2.21,2.75,3.05,1.78,2125.87,1.00


In [65]:
train_df[train_df['Class']==1][train_df['Amount']==0].describe()

C:\Users\mega\AppData\Local\Temp\ipykernel_3172\484607969.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train_df[train_df['Class']==1][train_df['Amount']==0].describe()


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
count,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00,27.00
mean,100142.78,-3.09,3.22,-5.85,5.41,-0.71,-1.61,-4.74,-0.19,-3.43,-5.35,4.10,-6.07,-0.32,-7.84,-0.40,-2.64,-5.04,-1.24,-0.32,0.73,-0.20,-0.03,0.02,-0.12,0.10,0.23,0.23,0.12,0.00,1.00
std,45302.16,5.99,4.46,5.25,2.11,4.75,1.98,8.04,7.91,2.47,5.06,2.49,3.85,1.38,4.37,1.04,3.47,6.45,2.60,1.57,2.04,4.29,1.74,1.22,0.41,0.70,0.41,1.28,0.42,0.00,0.00
min,406.00,-28.26,-8.40,-26.87,2.05,-18.00,-5.77,-41.51,-38.99,-13.43,-24.40,-1.70,-13.21,-3.08,-13.78,-2.84,-11.29,-20.58,-7.55,-3.68,-0.71,-21.45,-0.98,-2.34,-1.05,-1.89,-0.26,-5.41,-0.88,0.00,1.00
25%,74394.50,-5.30,1.69,-6.45,3.99,-2.23,-2.33,-5.72,0.28,-4.18,-7.19,2.79,-9.45,-1.11,-10.48,-0.56,-5.48,-10.95,-3.51,-1.57,-0.02,0.13,-0.81,-0.33,-0.41,-0.10,0.02,0.08,-0.15,0.00,1.00
50%,102669.00,-1.05,3.14,-5.87,5.47,-0.42,-1.82,-2.54,0.71,-3.16,-4.23,4.44,-4.60,-0.31,-9.86,-0.24,-2.48,-4.93,-0.96,-0.59,0.33,0.42,-0.47,-0.07,-0.06,0.35,0.07,0.49,0.23,0.00,1.00
75%,138944.50,0.60,3.58,-2.54,5.87,1.63,-1.12,-0.92,1.37,-1.96,-2.88,6.31,-3.82,0.31,-4.75,0.38,0.42,0.26,1.10,1.00,0.55,1.07,-0.05,0.04,0.14,0.39,0.31,0.77,0.36,0.00,1.00
max,165981.00,1.20,21.47,0.06,11.74,9.88,6.07,1.30,6.43,-0.78,2.69,7.19,-0.05,2.73,0.98,1.04,2.13,6.44,2.59,3.12,10.44,2.57,8.36,4.91,0.67,1.20,1.17,1.37,0.73,0.00,1.00


In [66]:
train_df['Amount'].value_counts().sort_index()

Amount
0.00        1825
0.01         718
0.02          85
0.03           3
0.04          11
            ... 
11898.09       1
12910.93       1
18910.00       1
19656.53       1
25691.16       1
Name: count, Length: 32767, dtype: int64

In [67]:
# 이상치 제거 
#이상치 체크.

def get_outlier(df=None, column=None, weight=1.5):
    # fraud에 해당하는 column 데이터만 추출, 1/4 분위와 3/4 분위 지점을 np.percentile로 구함.
    fraud = df[df['Class']==1][column]
    quantile_25 = np.percentile(fraud.values, 25)
    quantile_75 = np.percentile(fraud.values, 75)
    # IQR을 구하고, IQR에 1.5를 곱하여 최대값과 최소값 지점 구함.
    iqr = quantile_75 - quantile_25
    iqr_weight = iqr * weight
    lowest_val = quantile_25 - iqr_weight
    highest_val = quantile_75 + iqr_weight
    # 최대값 보다 크거나, 최소값 보다 작은 값을 아웃라이어로 설정하고 DataFrame index 반환.
    outlier_index = fraud[(fraud < lowest_val) | (fraud > highest_val)].index
    return outlier_index

#실제 14에 있는 이상치 체크
outlier_index = get_outlier(df=train_df, column='V14', weight=1.5)
print('이상치 데이터 인덱스:', outlier_index)




이상치 데이터 인덱스: Index([8296, 8615, 9035, 9252], dtype='int64')


In [68]:
# amount scaling, V14이상치 제거
from sklearn.preprocessing import StandardScaler, RobustScaler

def get_preprocessed_df(df=None):
    df_copy = df.copy()
    rob_scaler = RobustScaler()
    amount_n = rob_scaler.fit_transform(df_copy['Amount'].values.reshape(-1, 1))
    # 변환된 Amount를 Amount_Scaled로 피처명 변경후 DataFrame맨 앞 컬럼으로 입력
    df_copy.insert(0, 'Amount_Scaled', amount_n)
    # 기존 Time, Amount 피처 삭제
    df_copy.drop(['Time','Amount'], axis=1, inplace=True)
    # 이상치 데이터 삭제하는 로직 추가
    outlier_index = get_outlier(df=df_copy, column='V14', weight=1.5)
    df_copy.drop(outlier_index, axis=0, inplace=True)
    return df_copy

train_df= get_preprocessed_df(train_df)


In [77]:
# print("전체 중복:", train_df.duplicated().sum())

# print(
#     train_df[train_df.duplicated(keep=False)]
#     ['Class']
#     .value_counts()
# )

In [69]:
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()


In [70]:
# 트레인 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    stratify=y_labels,
    random_state=42
)

In [71]:
#다시 데이터 프레임 재구성
train_sample_df = X_train.copy()
train_sample_df['Class'] = y_train

# 사기/일반인 데이터 프레임만 따로 구분
fraud_train = train_sample_df[
    train_sample_df['Class'] == 1
].copy()

normal_train = train_sample_df[
    train_sample_df['Class'] == 0
].copy()



In [72]:
fraud_count = len(fraud_train)
normal_chunk_size = fraud_count * 10

print("Fraud 개수:", fraud_count)
print("한 묶음당 Normal 개수:", normal_chunk_size)

Fraud 개수: 390
한 묶음당 Normal 개수: 3900


In [73]:
normal_train = normal_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [74]:
sample_datasets = []

for start in range(0, len(normal_train), normal_chunk_size):
    normal_chunk = normal_train.iloc[
        start:start + normal_chunk_size
    ]

    # 마지막에 3930개보다 적게 남은 데이터는 일단 제외
    if len(normal_chunk) < normal_chunk_size:
        break

    sampled_df = pd.concat([
        fraud_train,
        normal_chunk
    ])

    sampled_df = sampled_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    sample_datasets.append(sampled_df)

print("만들어진 샘플 수:", len(sample_datasets))

만들어진 샘플 수: 58


In [75]:
# 로지스틱 회귀 사용(체크용)

from sklearn.linear_model import LogisticRegression

models = []

for sampled_df in sample_datasets:

    X_sample = sampled_df.drop(columns='Class')
    y_sample = sampled_df['Class']

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(X_sample, y_sample)

    models.append(model)

print("학습된 모델 수:", len(models))

학습된 모델 수: 58


In [76]:
pred_proba_list = []

for model in models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    pred_proba_list.append(pred_proba)

In [79]:
pred_proba_list = []

for model in models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    pred_proba_list.append(pred_proba)

In [80]:
mean_pred_proba = np.mean(
    pred_proba_list,
    axis=0
)

In [81]:
final_pred = (
    mean_pred_proba >= 0.5
).astype(int)

In [82]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

print(confusion_matrix(y_test, final_pred))

print(
    classification_report(
        y_test,
        final_pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        mean_pred_proba
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        mean_pred_proba
    )
)

[[56727   136]
 [    9    89]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.40      0.91      0.55        98

    accuracy                           1.00     56961
   macro avg       0.70      0.95      0.77     56961
weighted avg       1.00      1.00      1.00     56961

ROC-AUC: 0.9898883352648165
PR-AUC: 0.7678310486402558


In [83]:
#XGB 실행 58개 앙상블

from xgboost import XGBClassifier
import numpy as np

xgb_models = []

for i, sampled_df in enumerate(sample_datasets):

    X_sample = sampled_df.drop(columns='Class')
    y_sample = sampled_df['Class']

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(X_sample, y_sample)

    xgb_models.append(xgb_model)

print("학습된 XGB 모델 수:", len(xgb_models))

학습된 XGB 모델 수: 58


In [84]:
#58개 모델의 예측확률 평균

xgb_pred_proba_list = []

for model in xgb_models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    xgb_pred_proba_list.append(pred_proba)

xgb_mean_pred_proba = np.mean(
    xgb_pred_proba_list,
    axis=0
)

In [85]:
from sklearn.metrics import roc_auc_score

xgb_roc_auc = roc_auc_score(
    y_test,
    xgb_mean_pred_proba
)

print("XGBoost ROC-AUC:", xgb_roc_auc)

XGBoost ROC-AUC: 0.9916096224114744


In [86]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    average_precision_score
)

xgb_final_pred = (
    xgb_mean_pred_proba >= 0.5
).astype(int)

print(confusion_matrix(
    y_test,
    xgb_final_pred
))

print(classification_report(
    y_test,
    xgb_final_pred
))

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        xgb_mean_pred_proba
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        xgb_mean_pred_proba
    )
)

[[56769    94]
 [   10    88]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56863
           1       0.48      0.90      0.63        98

    accuracy                           1.00     56961
   macro avg       0.74      0.95      0.81     56961
weighted avg       1.00      1.00      1.00     56961

ROC-AUC: 0.9916096224114744
PR-AUC: 0.879950170091397


In [ ]:
# 중복된 행 제거(단, 이는 의도적일 수 있으니, 체크하고 사용하시길.)
before = len(train_df)

train_df.drop_duplicates(inplace=True)
train_df.reset_index(drop=True, inplace=True)

after = len(train_df)

print(f"제거된 중복 행 수: {before - after}")
print(f"제거 후 데이터 수: {after}")

제거된 중복 행 수: 9144
제거 후 데이터 수: 275659


In [ ]:
y_labels=train_df.iloc[:, -1].copy()
X_features= train_df.drop(columns=['Class']).copy()


In [89]:
# 트레인 
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_features,
    y_labels,
    test_size=0.2,
    stratify=y_labels,
    random_state=42
)

In [90]:
#다시 데이터 프레임 재구성
train_sample_df = X_train.copy()
train_sample_df['Class'] = y_train

# 사기/일반인 데이터 프레임만 따로 구분
fraud_train = train_sample_df[
    train_sample_df['Class'] == 1
].copy()

normal_train = train_sample_df[
    train_sample_df['Class'] == 0
].copy()



In [91]:
fraud_count = len(fraud_train)
normal_chunk_size = fraud_count * 10

print("Fraud 개수:", fraud_count)
print("한 묶음당 Normal 개수:", normal_chunk_size)

Fraud 개수: 375
한 묶음당 Normal 개수: 3750


In [92]:
normal_train = normal_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

In [93]:
sample_datasets = []

for start in range(0, len(normal_train), normal_chunk_size):
    normal_chunk = normal_train.iloc[
        start:start + normal_chunk_size
    ]

    # 마지막에 3930개보다 적게 남은 데이터는 일단 제외
    if len(normal_chunk) < normal_chunk_size:
        break

    sampled_df = pd.concat([
        fraud_train,
        normal_chunk
    ])

    sampled_df = sampled_df.sample(
        frac=1,
        random_state=42
    ).reset_index(drop=True)

    sample_datasets.append(sampled_df)

print("만들어진 샘플 수:", len(sample_datasets))

만들어진 샘플 수: 58


In [94]:
# 로지스틱 회귀 사용(체크용)

from sklearn.linear_model import LogisticRegression

models = []

for sampled_df in sample_datasets:

    X_sample = sampled_df.drop(columns='Class')
    y_sample = sampled_df['Class']

    model = LogisticRegression(
        max_iter=1000,
        random_state=42
    )

    model.fit(X_sample, y_sample)

    models.append(model)

print("학습된 모델 수:", len(models))

학습된 모델 수: 58


In [95]:
pred_proba_list = []

for model in models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    pred_proba_list.append(pred_proba)

In [96]:
pred_proba_list = []

for model in models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    pred_proba_list.append(pred_proba)

In [97]:
mean_pred_proba = np.mean(
    pred_proba_list,
    axis=0
)

In [98]:
final_pred = (
    mean_pred_proba >= 0.5
).astype(int)

In [99]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

print(confusion_matrix(y_test, final_pred))

print(
    classification_report(
        y_test,
        final_pred
    )
)

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        mean_pred_proba
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        mean_pred_proba
    )
)

[[54883   155]
 [   12    82]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     55038
           1       0.35      0.87      0.50        94

    accuracy                           1.00     55132
   macro avg       0.67      0.93      0.75     55132
weighted avg       1.00      1.00      1.00     55132

ROC-AUC: 0.9727816680622209
PR-AUC: 0.7526933316759082


In [100]:
#XGB 실행 58개 앙상블

from xgboost import XGBClassifier
import numpy as np

xgb_models = []

for i, sampled_df in enumerate(sample_datasets):

    X_sample = sampled_df.drop(columns='Class')
    y_sample = sampled_df['Class']

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='auc',
        random_state=42,
        n_jobs=-1
    )

    xgb_model.fit(X_sample, y_sample)

    xgb_models.append(xgb_model)

print("학습된 XGB 모델 수:", len(xgb_models))

학습된 XGB 모델 수: 58


In [101]:
#58개 모델의 예측확률 평균

xgb_pred_proba_list = []

for model in xgb_models:
    pred_proba = model.predict_proba(X_test)[:, 1]
    xgb_pred_proba_list.append(pred_proba)

xgb_mean_pred_proba = np.mean(
    xgb_pred_proba_list,
    axis=0
)

In [102]:
from sklearn.metrics import roc_auc_score

xgb_roc_auc = roc_auc_score(
    y_test,
    xgb_mean_pred_proba
)

print("XGBoost ROC-AUC:", xgb_roc_auc)

XGBoost ROC-AUC: 0.9800082032297994


In [103]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    average_precision_score
)

xgb_final_pred = (
    xgb_mean_pred_proba >= 0.5
).astype(int)

print(confusion_matrix(
    y_test,
    xgb_final_pred
))

print(classification_report(
    y_test,
    xgb_final_pred
))

print(
    "ROC-AUC:",
    roc_auc_score(
        y_test,
        xgb_mean_pred_proba
    )
)

print(
    "PR-AUC:",
    average_precision_score(
        y_test,
        xgb_mean_pred_proba
    )
)

[[54975    63]
 [   14    80]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     55038
           1       0.56      0.85      0.68        94

    accuracy                           1.00     55132
   macro avg       0.78      0.92      0.84     55132
weighted avg       1.00      1.00      1.00     55132

ROC-AUC: 0.9800082032297994
PR-AUC: 0.8331501475737263


In [104]:
train_check = X_train.copy()
test_check = X_test.copy()

train_rows = set(map(tuple, train_check.values))
test_rows = set(map(tuple, test_check.values))

overlap = train_rows.intersection(test_rows)

print("Train/Test 동일 Feature 행 수:", len(overlap))

Train/Test 동일 Feature 행 수: 0
